# Interactive docking box

Adjust the docking search box in Mol* — resize, recenter, or rotate it — then
dock with the updated geometry.

Open **Settings → Docking Box** in the viewer after running the interactive cell
below. When you release the mouse or slider, the box geometry syncs back to your
`Docking` object. Re-run the inspect cells to see the updated pocket and payload.

This notebook uses bundled BRD4 demo structures (`BRD_DATA_DIR`). No platform sync
is required until you run docking at the end.

In [ ]:
from dotenv import load_dotenv

load_dotenv()

In [ ]:
%load_ext autoreload
%autoreload 2

## Setup

In [ ]:
from deeporigin.drug_discovery import BRD_DATA_DIR, Docking, Ligand, Pocket, Protein
from deeporigin.drug_discovery.docking_common import resolve_docking_box_geometry

## Protein, pocket, and ligand

The pocket is centered on the co-crystal ligand binding site. Set the initial
search box size before opening the interactive viewer.

In [ ]:
protein = Protein.from_file(BRD_DATA_DIR / "brd.pdb")

ligand = Ligand.from_sdf(BRD_DATA_DIR / "brd-2.sdf")

pocket = Pocket.from_ligand(ligand, name="binding-site")
pocket.box_size_x = pocket.box_size_y = pocket.box_size_z = 15.0

box_center, box_size = resolve_docking_box_geometry(pocket)
print(f"Box center: {box_center}")
print(f"Box size:   {box_size}")

docking = Docking(protein=protein, pocket=pocket, ligand=ligand)
docking

## Interactive box viewer

1. Run the cell below.
2. In Mol*, open **Settings → Docking Box**.
3. Resize, recenter, or rotate the box.
4. Release the mouse or slider.
5. Re-run the inspect cells below.

In [ ]:
handle = docking.show_box(interactive=True)

## Inspect committed geometry

Re-run this cell after editing the box.

In [ ]:
print(f"Pocket center: {docking.pocket.center}")
print(
    "Pocket box size:",
    [
        docking.pocket.box_size_x,
        docking.pocket.box_size_y,
        docking.pocket.box_size_z,
    ],
)
print(f"rotation_deg: {docking.rotation_deg}")

if handle.committed is not None:
    print(f"Last commit: {handle.committed}")

## Preview tool inputs

Shows the pocket block that `docking.run()` would send. Does not call the platform.

In [ ]:
params, _metadata = docking._build_tool_inputs()
params["pocket"]

## Run docking (platform)

Adjust the box above first, then re-run the inspect cells to confirm the geometry.
Requires `deeporigin login` and a billing-enabled org.

In [ ]:
docking.protein.upload()

docking.run(quote=True)
docking.estimate

In [ ]:
poses = docking.run()
poses

## View docked poses

Pose overlays are not supported in `interactive=True` mode, so use a static viewer here.

In [ ]:
poses.download()
docking.show_box(poses=[poses[0]])